# 少ない枚数で支払う ― Optimize で最適化します

## この章で学ぶこと

Z3 が解く問題には、2種類あります。

- **制約充足問題**：条件を満たす解を1つ見つけます
- **最適化問題**：条件を満たす解の中で、最も良いものを探索します

本章では、同じ「コインの枚数」問題を両方の方法で解き、どのような差異があるかを確認します。

- `Solver` で解を収集して比較します（枚数を数える必要があります）
- `Optimize` で「枚数が最小の解」を直接求めます
- `lower()` / `upper()` で「これが真の最良値である」ことを確認します

## 問題

134セントをできるだけ少ない枚数のコインで支払うには、どのように組み合わせればよいでしょうか。

- 使用可能なコイン：1, 2, 5, 10, 25, 50 セント
- 作成する金額：134 セント
- 目的：**使用するコインの枚数をできるだけ少なくします**

「合計を134セントにすること」が制約であり、「枚数を最小にすること」が目的です。Z3 では、この2つを別々に記述します。この点が本章の要点です。

## まず Solver で条件を満たす解をすべて収集します

`Solver` は「条件を満たす解を見つける」ための道具です。最少の枚数を確認するため、今回は条件を満たす解をすべて取り出し、枚数を自ら比較します。

In [1]:
from z3 import *
from helper.z3 import all_solutions   # 本書で用意している「全解を順に返す」関数

coins = [1, 2, 5, 10, 25, 50]   # 使えるコインの種類（セント）
total = 134                     # 作りたい金額
max_coins = 8                   # とりあえず「8枚以下」にしぼっておく

counts = [Int(f"n_{c}") for c in coins]                # 各コインの枚数
amount = Sum([c * n for c, n in zip(coins, counts)])   # 合計金額
num_coins = Sum(counts)                               # 合計枚数

solver = Solver()
for n in counts:
    solver.add(n >= 0)              # 枚数は 0 以上
solver.add(amount == total)         # 合計はちょうど 134 セント
solver.add(num_coins <= max_coins)  # 枚数は 8 枚以下


def show(model):
    """解を「8枚: 2セント×2, 5セント×1, ...」の形で表示する"""
    used = {c: model.eval(n).as_long() for c, n in zip(coins, counts)}
    used = {c: k for c, k in used.items() if k > 0}   # 0枚のコインは省く
    print(f"{model.eval(num_coins)}枚: " + ", ".join(f"{c}セント×{k}" for c, k in used.items()))


for model in all_solutions(solver):
    show(model)

8枚: 2セント×2, 5セント×1, 25セント×5
8枚: 1セント×2, 2セント×1, 10セント×3, 50セント×2
8枚: 1セント×1, 2セント×4, 25セント×1, 50セント×2
8枚: 1セント×4, 5セント×1, 25セント×1, 50セント×2
7枚: 2セント×2, 5セント×1, 25セント×3, 50セント×1
8枚: 2セント×2, 10セント×3, 25セント×2, 50セント×1
8枚: 1セント×2, 2セント×1, 5セント×1, 25セント×3, 50セント×1
7枚: 2セント×2, 10セント×3, 50セント×2
8枚: 2セント×2, 5セント×2, 10セント×2, 50セント×2
7枚: 1セント×2, 2セント×1, 5セント×1, 25セント×1, 50セント×2
6枚: 2セント×2, 5セント×1, 25セント×1, 50セント×2


条件を満たす解は **11通り** 得られました。最も少ないのは **6枚** の組み合わせです。

ただし、この方法には課題があります。

- `Solver` は「枚数が少ない順」に探索しているわけではありません。今回は最後に出力されましたが、その順序には保証がありません。
- すべての解を収集して比較するため、解が多いと時間がかかります。実際、`max_coins` の上限を設定しない場合、解は **11,713通り** になります。
- 上限の8枚も、本章で暫定的に決定した値です。「6枚より少なく支払う方法は存在しない」ことを証明したわけではありません。

「最も良い解を探索する」ための道具が `Optimize` です。

## Optimize で「枚数が最小の解」を直接求めます

`Optimize` は、条件を満たす解の中で**目的関数**（ここでは枚数）を最小・最大にする解を探索します。使用方法は `Solver` とほぼ同一であり、`minimize()` を追加するだけです。

In [2]:
opt = Optimize()

for n in counts:
    opt.add(n >= 0)
opt.add(amount == total)        # 金額の条件は同じ。枚数の上限はもういらない

obj = opt.minimize(num_coins)   # 目的関数：合計枚数を最小にする

print(opt.check())

model = opt.model()
show(model)
print("合計:", sum(c * model.eval(n).as_long() for c, n in zip(coins, counts)), "セント")

sat
6枚: 2セント×2, 5セント×1, 25セント×1, 50セント×2
合計: 134 セント


同じ6枚が答えとして得られました。`minimize()` は「目的関数のハンドル」を返します。このハンドルを用いると、最良値の保証範囲を確認できます。

In [3]:
print("最良値の下限:", opt.lower(obj))
print("最良値の上限:", opt.upper(obj))

# 少し悪くして比べてみる（上限を 7 にすると、6 枚の解は選べない）
opt.add(num_coins >= 7)
opt.check()
print("7枚以上の条件で:", opt.lower(obj), opt.upper(obj))

最良値の下限: 6
最良値の上限: 6
7枚以上の条件で: 7 7


`lower()` と `upper()` は「最良値がこの範囲にある」ことを保証します。最初の結果では下限と上限がともに 6 であるため、**6枚より少なく支払う方法は存在しない**ことを Z3 が示したことになります。

2つ目のように条件を追加すると、範囲は 7 に変化します。`Optimize` は条件が変われば、その都度、最良値を再探索します。

`Solver` では「見つかった中で最も良いもの」しか述べることはできませんでしたが、`Optimize` は「これが最良である」と述べることができます。この点が2つの大きな差異です。

| | `Solver` | `Optimize` |
|---|---|---|
| 目的 | 条件を満たす解を1つ見つけます | 目的関数が最良の解を見つけます |
| 制約の追加 | `solver.add()` | `opt.add()` |
| 最適化 | できません | `opt.minimize()` / `opt.maximize()` |
| 得られる答え | いずれか1つ（最良とは限りません） | 最良の1つ |
| 最良である保証 | ありません | `lower()` / `upper()` で確認できます |

## 関数化し、コインの種類を変更します

同形式の問題は、関数化することで再利用できます。さらに、日本円でも試します。

In [4]:
def min_coin_counts(coin_types, amount):
    """金額 amount を最少枚数で作る組み合わせを返す"""
    counts = [Int(f"n_{c}") for c in coin_types]
    opt = Optimize()
    for n in counts:
        opt.add(n >= 0)
    opt.add(Sum([c * n for c, n in zip(coin_types, counts)]) == amount)
    opt.minimize(Sum(counts))
    assert opt.check() == sat
    model = opt.model()
    return {c: model[n].as_long() for c, n in zip(coin_types, counts) if model[n].as_long() > 0}


print(min_coin_counts([1, 2, 5, 10, 25, 50], 134))
print(min_coin_counts([1, 5, 10, 50, 100, 500], 1340))

{2: 2, 5: 1, 25: 1, 50: 2}
{10: 4, 100: 3, 500: 2}


コインの種類と金額を変更するだけで、同じコードを利用できます。1340円の場合、500円×2 + 100円×3 + 10円×4 の9枚が最少であるという答えが得られました。

## まとめ

- 制約充足問題は「条件を満たす解を探索する」問題であり、最適化問題は「条件を満たす中で最も良い解を探索する」問題です
- `Solver` は解を1つ（または全部）見つけますが、最良であるかどうかは判定できません
- `Optimize` は `minimize()` / `maximize()` で目的関数を指定し、最良の解を探索します
- `lower()` と `upper()` が一致すれば、それが真の最良値です
- 解をすべて収集する方法は、解が多いと直ちに扱いが困難になります（この問題では上限を設定しない場合、11,713通り）